In [4]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw/archive")

customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
from pathlib import Path

for file in DATA_DIR.glob("*.csv"):
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [6]:
files = list(DATA_DIR.glob("*.csv"))

print(f"Number of datasets: {len(files)}")

Number of datasets: 9


In [7]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/archive")

datasets = {}

for file in DATA_DIR.glob("*.csv"):
    datasets[file.stem] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets")

Loaded 9 datasets


In [8]:
datasets.keys()

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [9]:
inventory = []

for name, df in datasets.items():
    inventory.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

inventory = pd.DataFrame(inventory)
inventory.sort_values("table")

,table,rows,columns
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
3,olist_order_items_dataset,112650,7
4,olist_order_payments_dataset,103886,5
5,olist_order_reviews_dataset,99224,7
2,olist_orders_dataset,99441,8
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


In [11]:
inventory.to_markdown("../docs/data_inventory.md", index=False)

In [12]:
for name, df in datasets.items():
    print("=" * 70)
    print(f"Dataset: {name}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(df.dtypes)
    print()

Dataset: olist_customers_dataset
Rows: 99441
Columns: 5
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Dataset: olist_geolocation_dataset
Rows: 1000163
Columns: 5
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Dataset: olist_orders_dataset
Rows: 99441
Columns: 8
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Dataset: olist_order_items_dataset
Rows: 112650
Columns: 7
order_id                   str
order_item_id            int64
prod

In [13]:
for name, df in datasets.items():

    print("=" * 70)
    print(f"Dataset : {name}")

    print(f"Rows : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    print(f"\nMissing Values")
    print(df.isnull().sum())

    print(f"\nDuplicate Rows")
    print(df.duplicated().sum())

    print("\n")


Dataset : olist_customers_dataset
Rows : 99441
Columns : 5

Missing Values
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate Rows
0


Dataset : olist_geolocation_dataset
Rows : 1000163
Columns : 5

Missing Values
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicate Rows
261831


Dataset : olist_orders_dataset
Rows : 99441
Columns : 8

Missing Values
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate Rows
0


Dataset : olist_order_items_dataset
Rows : 112650
C

In [14]:
quality_summary = []

for name, df in datasets.items():
    total_missing = int(df.isnull().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    quality_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": total_missing,
        "duplicate_rows": duplicate_rows,
        "status": "Needs Review" if total_missing > 0 or duplicate_rows > 0 else "Clean"
    })

quality_summary = pd.DataFrame(quality_summary)

quality_summary.sort_values("dataset").reset_index(drop=True)

,dataset,rows,columns,missing_values,duplicate_rows,status
0,olist_customers_dataset,99441,5,0,0,Clean
1,olist_geolocation_dataset,1000163,5,0,261831,Needs Review
2,olist_order_items_dataset,112650,7,0,0,Clean
3,olist_order_payments_dataset,103886,5,0,0,Clean
4,olist_order_reviews_dataset,99224,7,145903,0,Needs Review
5,olist_orders_dataset,99441,8,4908,0,Needs Review
6,olist_products_dataset,32951,9,2448,0,Needs Review
7,olist_sellers_dataset,3095,4,0,0,Clean
8,product_category_name_translation,71,2,0,0,Clean


In [15]:
for name, df in datasets.items():

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("=" * 70)
        print(name)
        print(missing)
        print()

olist_orders_dataset
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

olist_order_reviews_dataset
review_comment_title      87656
review_comment_message    58247
dtype: int64

olist_products_dataset
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



In [16]:
for name, df in datasets.items():
    print("=" * 70)
    print(name)
    print(df.columns.tolist())
    print()

olist_customers_dataset
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_geolocation_dataset
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

olist_orders_dataset
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_order_payments_dataset
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

olist_order_reviews_dataset
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

olist_products_dataset
['product_id', 'product_category_name', 'product_name_lenght', 'product_d

In [17]:
primary_key_checks = {
    "olist_customers_dataset": ["customer_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "product_category_name_translation": ["product_category_name"],
}

pk_results = []

for table_name, key_columns in primary_key_checks.items():
    df = datasets[table_name]

    pk_results.append({
        "table": table_name,
        "candidate_key": " + ".join(key_columns),
        "null_values": int(df[key_columns].isnull().sum().sum()),
        "duplicate_keys": int(df.duplicated(subset=key_columns).sum()),
        "is_valid": (
            df[key_columns].isnull().sum().sum() == 0
            and df.duplicated(subset=key_columns).sum() == 0
        )
    })

pk_results = pd.DataFrame(pk_results)
pk_results

,table,candidate_key,null_values,duplicate_keys,is_valid
0,olist_customers_dataset,customer_id,0,0,True
1,olist_orders_dataset,order_id,0,0,True
2,olist_products_dataset,product_id,0,0,True
3,olist_sellers_dataset,seller_id,0,0,True
4,olist_order_items_dataset,order_id + order_item_id,0,0,True
5,olist_order_payments_dataset,order_id + payment_sequential,0,0,True
6,olist_order_reviews_dataset,review_id + order_id,0,0,True
7,product_category_name_translation,product_category_name,0,0,True


In [18]:
foreign_key_checks = [
    {
        "relationship": "orders → customers",
        "child_table": "olist_orders_dataset",
        "child_column": "customer_id",
        "parent_table": "olist_customers_dataset",
        "parent_column": "customer_id",
    },
    {
        "relationship": "order_items → orders",
        "child_table": "olist_order_items_dataset",
        "child_column": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_column": "order_id",
    },
    {
        "relationship": "payments → orders",
        "child_table": "olist_order_payments_dataset",
        "child_column": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_column": "order_id",
    },
    {
        "relationship": "reviews → orders",
        "child_table": "olist_order_reviews_dataset",
        "child_column": "order_id",
        "parent_table": "olist_orders_dataset",
        "parent_column": "order_id",
    },
    {
        "relationship": "order_items → products",
        "child_table": "olist_order_items_dataset",
        "child_column": "product_id",
        "parent_table": "olist_products_dataset",
        "parent_column": "product_id",
    },
    {
        "relationship": "order_items → sellers",
        "child_table": "olist_order_items_dataset",
        "child_column": "seller_id",
        "parent_table": "olist_sellers_dataset",
        "parent_column": "seller_id",
    },
    {
        "relationship": "products → category translation",
        "child_table": "olist_products_dataset",
        "child_column": "product_category_name",
        "parent_table": "product_category_name_translation",
        "parent_column": "product_category_name",
    },
]

fk_results = []

for check in foreign_key_checks:
    child_values = datasets[check["child_table"]][check["child_column"]]
    parent_values = datasets[check["parent_table"]][check["parent_column"]]

    non_null_child_values = child_values.dropna()

    orphan_mask = ~non_null_child_values.isin(parent_values)
    orphan_count = int(orphan_mask.sum())

    fk_results.append({
        "relationship": check["relationship"],
        "child_column": check["child_column"],
        "parent_column": check["parent_column"],
        "non_null_child_values": len(non_null_child_values),
        "orphan_records": orphan_count,
        "is_valid": orphan_count == 0,
    })

fk_results = pd.DataFrame(fk_results)

fk_results

,relationship,child_column,parent_column,non_null_child_values,orphan_records,is_valid
0,orders → customers,customer_id,customer_id,99441,0,True
1,order_items → orders,order_id,order_id,112650,0,True
2,payments → orders,order_id,order_id,103886,0,True
3,reviews → orders,order_id,order_id,99224,0,True
4,order_items → products,product_id,product_id,112650,0,True
5,order_items → sellers,seller_id,seller_id,112650,0,True
6,products → category translation,product_category_name,product_category_name,32341,13,False


In [19]:
products_df = datasets["olist_products_dataset"]
translation_df = datasets["product_category_name_translation"]

valid_categories = translation_df["product_category_name"]

orphan_category_rows = products_df[
    products_df["product_category_name"].notna()
    & ~products_df["product_category_name"].isin(valid_categories)
]

orphan_category_summary = (
    orphan_category_rows["product_category_name"]
    .value_counts()
    .rename_axis("product_category_name")
    .reset_index(name="product_count")
)

orphan_category_summary

,product_category_name,product_count
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


In [20]:
print("Tables in Olist Dataset:\n")

for name in sorted(datasets.keys()):
    print("-", name)

Tables in Olist Dataset:

- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- product_category_name_translation


In [21]:
table_description = {
    "olist_customers_dataset": "Customer information",
    "olist_orders_dataset": "Order information",
    "olist_order_items_dataset": "Products inside each order",
    "olist_products_dataset": "Product information",
    "olist_sellers_dataset": "Seller information",
    "olist_order_payments_dataset": "Payment information",
    "olist_order_reviews_dataset": "Customer reviews",
    "olist_geolocation_dataset": "Zip code geolocation",
    "product_category_name_translation": "Category translation (Portuguese → English)"
}

for table, description in table_description.items():
    print("=" * 70)
    print(f"Table : {table}")
    print(f"Purpose : {description}")
    print()

Table : olist_customers_dataset
Purpose : Customer information

Table : olist_orders_dataset
Purpose : Order information

Table : olist_order_items_dataset
Purpose : Products inside each order

Table : olist_products_dataset
Purpose : Product information

Table : olist_sellers_dataset
Purpose : Seller information

Table : olist_order_payments_dataset
Purpose : Payment information

Table : olist_order_reviews_dataset
Purpose : Customer reviews

Table : olist_geolocation_dataset
Purpose : Zip code geolocation

Table : product_category_name_translation
Purpose : Category translation (Portuguese → English)



In [22]:
relationships = {
    "olist_customers_dataset": [],
    "olist_orders_dataset": ["olist_customers_dataset"],
    "olist_order_items_dataset": [
        "olist_orders_dataset",
        "olist_products_dataset",
        "olist_sellers_dataset"
    ],
    "olist_order_payments_dataset": ["olist_orders_dataset"],
    "olist_order_reviews_dataset": ["olist_orders_dataset"],
    "olist_products_dataset": [
        "product_category_name_translation"
    ],
    "olist_sellers_dataset": [],
    "olist_geolocation_dataset": [],
    "product_category_name_translation": []
}

for table, parents in relationships.items():
    print("=" * 70)
    print(f"Table : {table}")

    if len(parents) == 0:
        print("Depends on : None")
    else:
        print("Depends on :")
        for parent in parents:
            print(f"   -> {parent}")

    print()

Table : olist_customers_dataset
Depends on : None

Table : olist_orders_dataset
Depends on :
   -> olist_customers_dataset

Table : olist_order_items_dataset
Depends on :
   -> olist_orders_dataset
   -> olist_products_dataset
   -> olist_sellers_dataset

Table : olist_order_payments_dataset
Depends on :
   -> olist_orders_dataset

Table : olist_order_reviews_dataset
Depends on :
   -> olist_orders_dataset

Table : olist_products_dataset
Depends on :
   -> product_category_name_translation

Table : olist_sellers_dataset
Depends on : None

Table : olist_geolocation_dataset
Depends on : None

Table : product_category_name_translation
Depends on : None

